# Harmful Augmentation: Solarize
Full baseline augmentations + Solarization (p=0.5) added on top.

**Expected runtime:** ~1.5h on T4.  
**Platform:** Google Colab — enable GPU via Runtime → Change runtime type → T4 GPU.

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
assert torch.cuda.is_available(), 'No GPU — enable GPU via Runtime → Change runtime type.'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
import os, subprocess

REPO_URL   = 'https://github.com/SAlaMusa/IDL.git'
REPO_DIR   = '/content/IDL'
OUTPUT_DIR = '/content/drive/MyDrive/simclr_ablations'

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
import glob, shutil, csv, datetime

EXP_NAME  = 'harmful_solarize'
CONFIG    = 'configs/harmful_solarize.yaml'
EPOCHS    = 200
BATCH     = 256
CKPT_PATH = os.path.join(OUTPUT_DIR, f'{EXP_NAME}_ep{EPOCHS}.pth.tar')
RESULTS   = os.path.join(OUTPUT_DIR, f'{EXP_NAME}_results.csv')

# Pretraining
if os.path.exists(CKPT_PATH):
    print('Checkpoint exists, skipping pretraining.')
else:
    subprocess.run([
        'python', 'run.py', '--config', CONFIG,
        '--epochs', str(EPOCHS), '--batch-size', str(BATCH),
        '-j', '2', '--seed', '42', '--fp16-precision',
    ], check=True)
    latest = max(glob.glob('runs/**/*.pth.tar', recursive=True), key=os.path.getmtime)
    shutil.copy2(latest, CKPT_PATH)
    print(f'Checkpoint saved: {CKPT_PATH}')

# Linear Evaluation
subprocess.run([
    'python', 'linear_eval.py',
    '--checkpoint', CKPT_PATH, '--dataset', 'cifar10',
    '--arch', 'resnet18', '--epochs', '100',
    '-b', '256', '-j', '2', '--seed', '42',
], check=True)

with open('linear_eval_results.csv') as f:
    best_top1 = float(list(csv.DictReader(f))[-1]['best_top1'])

with open(RESULTS, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['exp_name', 'config', 'best_top1', 'checkpoint', 'timestamp'])
    w.writerow([EXP_NAME, CONFIG, f'{best_top1:.2f}', CKPT_PATH,
                datetime.datetime.now().strftime('%Y-%m-%d %H:%M')])

print(f'\nBest Top-1: {best_top1:.2f}%')
print(f'Results saved to: {RESULTS}')